# 04p2 — NDMM PBMC QC Plots: Checking frequencies across celltypes
#### By [Mansi Singh](mailto:mansi.singh@alleninstitute.org), Comp Bio, Allen Institute for Immunology

**Aim:** Generate quality control visualizations showing cell type proportions across experimental pools at L1, L2, and L3 annotation levels for NDMM PBMC samples (193 samples, doublets removed).



In [ ]:
# =============================================================================
# Import Libraries
# =============================================================================
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=RuntimeWarning)

# Core data libraries
import pandas as pd
import numpy as np

# Single-cell analysis
import scanpy as sc
import anndata as ad

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go

# System/IO
import os
import hisepy

## 1. Load and Filter Data
Load the PBMC metadata and remove doublet cells.

In [ ]:
# Load NDMM PBMC metadata containing 193 samples with doublet annotations
# Data includes cell type annotations at L1, L2, L3 levels
sc_meta = pd.read_csv('../../../data/rna/NDMM_pbmc_193samples_doublet_metadata_all_types.csv.gz')

In [ ]:
# Filter out manually identified doublets
# Decision: Keep only cells marked as 'no' in doublets_manual column
sc_meta = sc_meta[sc_meta['doublets_manual'] == 'no']

# Verify filtered dataset dimensions
print(f"Filtered dataset shape: {sc_meta.shape[0]} cells, {sc_meta.shape[1]} columns")

In [ ]:
# Display available columns for reference
sc_meta.columns

## 2. Cell Proportion Visualizations (Matplotlib)
Stacked bar charts showing cell type proportions per pool at L1, L2, and L3 annotation levels.

In [ ]:
def plot_cell_proportions_matplotlib(metadata_df, annotation_col, level_name, figsize=(25, 10)):
    """
    Generate a stacked bar chart of cell type proportions by pool.
    
    Parameters:
    -----------
    metadata_df : pd.DataFrame
        Metadata dataframe with cell annotations
    annotation_col : str
        Column name for cell type annotations (e.g., 'tidy.aifi_l1')
    level_name : str
        Label for the annotation level (e.g., 'L1')
    figsize : tuple
        Figure dimensions
    """
    # Calculate cell proportions per pool (normalized by column)
    cell_proportion_df = pd.crosstab(
        metadata_df[annotation_col],
        metadata_df['pool_id'],
        normalize='columns'
    ).T
    
    # Create stacked bar plot
    ax = cell_proportion_df.plot(kind='bar', stacked=True, figsize=figsize)
    
    # Format axes
    ax.set_xlabel('Pool ID', fontsize=22)
    ax.set_ylabel('Proportion', fontsize=22)
    ax.tick_params(axis='x', labelsize=18)
    ax.tick_params(axis='y', labelsize=18)
    
    # Position legend outside plot
    ax.legend(
        title=f"{level_name} Cell Types",
        title_fontsize=22,
        fontsize=18,
        bbox_to_anchor=(1.05, 1),
        loc='upper left',
        borderaxespad=0.
    )
    
    ax.set_title(f"Cell Proportion by Pool ({level_name})", fontsize=24)
    ax.grid(False)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Generate L1, L2, L3 proportion plots using matplotlib
plot_cell_proportions_matplotlib(sc_meta, 'tidy.aifi_l1', 'L1')
plot_cell_proportions_matplotlib(sc_meta, 'tidy.aifi_l2', 'L2')
plot_cell_proportions_matplotlib(sc_meta, 'tidy.aifi_l3', 'L3', figsize=(25, 15))

## 3. Interactive Cell Proportion Visualizations (Plotly)
Interactive versions of the stacked bar charts for exploration.

In [ ]:
def plot_cell_proportions_plotly(metadata_df, annotation_col, level_name):
    """
    Generate an interactive stacked bar chart using Plotly.
    
    Parameters:
    -----------
    metadata_df : pd.DataFrame
        Metadata dataframe with cell annotations
    annotation_col : str
        Column name for cell type annotations
    level_name : str
        Label for the annotation level
    """
    # Calculate cell proportions per pool
    cell_proportion_df = pd.crosstab(
        metadata_df[annotation_col],
        metadata_df['pool_id'],
        normalize='columns'
    ).T
    
    # Build stacked bar figure
    fig = go.Figure()
    
    for col in cell_proportion_df.columns:
        fig.add_bar(
            name=col,
            x=cell_proportion_df.index,
            y=cell_proportion_df[col]
        )
    
    # Apply layout settings
    fig.update_layout(
        barmode='stack',
        title=f'Cell Proportion by Pool ({level_name})',
        xaxis=dict(
            title=dict(text='Pool ID', font=dict(size=22)),
            tickfont=dict(size=18)
        ),
        yaxis=dict(
            title=dict(text='Proportion', font=dict(size=22)),
            tickfont=dict(size=18)
        ),
        legend=dict(
            title=f'{level_name} Cell Types',
            title_font=dict(size=22),
            font=dict(size=18),
            x=1.02, y=1,
            xanchor='left', yanchor='top'
        ),
        margin=dict(l=40, r=200, t=80, b=80),
        height=600,
        width=1200,
    )
    
    fig.show()

In [ ]:
# Generate interactive L1, L2, L3 proportion plots
plot_cell_proportions_plotly(sc_meta, 'tidy.aifi_l1', 'L1')
plot_cell_proportions_plotly(sc_meta, 'tidy.aifi_l2', 'L2')
plot_cell_proportions_plotly(sc_meta, 'tidy.aifi_l3', 'L3')

## 4. Environment Information (Debug)

In [ ]:
# Print environment info for reproducibility
import sys
print(f"Python executable: {sys.executable}")
print(f"Pandas version: {pd.__version__}")
print(f"Scanpy version: {sc.__version__}")